# Train the Khmer OCR recognizer

Works unmodified on **Colab**, **Kaggle**, and a **local machine** -- on **CPU, GPU, or TPU**:
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `recognizer/checkpoints/` in the repo.

The accelerator (TPU / GPU / CPU) is auto-detected -- no notebook changes needed either
way. **To actually get one**, select it as the runtime/accelerator in Colab
(Runtime > Change runtime type > GPU or TPU) or Kaggle (Settings > Accelerator >
GPU or TPU) *before* running this notebook; both platforms ship `torch_xla`
preinstalled on their TPU runtimes, so no extra install step is needed here. Detection
covers both TPU generations -- the legacy XRT env vars *and* the PJRT ones every
current runtime uses -- so section 1 prints the device it actually resolved
(`torch device: cuda` / `xla:0` / `cpu`); check that line before starting the run
rather than inferring it later from step timings.

On GPU, batch size is auto-probed to fit the available VRAM (OOM-probing auto-tune);
on TPU and CPU that probe is skipped -- XLA compiles lazily and never raises a
catchable Python OOM -- and `TrainConfig.batch_size` is used exactly as configured, so
set it yourself if you hit a TPU HBM limit.

This pulls a **real, at-scale dataset** by default (all of `deepcopy_khmer_text_recognition`
/ `darayut_scene_text` / `sokheng_synthetic_v1`, plus 100k rows of
`chanrith_ocr_image_line` -- see section 2), roughly ~9GB on disk -- this is a
production-scale training run, not the small notebook-scale loop in
`notebooks/train_diagnose_eval.ipynb`. Use that other notebook first if you just want
to iterate quickly on model/training-loop changes.

Training logs every 100 steps and pushes a checkpoint to the `Panhapich/tuna-ocr`
Hugging Face repo (created private) every 10,000 steps.

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left sidebar;
Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`).

**TPU caveats** (GPU is the better-tested path here; TPU is supported but not tuned):
- PyTorch/XLA's support for the CTC loss op has historically been inconsistent across
  versions -- if training errors out or looks unusually slow on TPU, check whether
  `torch.nn.functional.ctc_loss` is silently falling back to a CPU path before assuming
  it's a bug in this repo.
- Batches here are variable-shaped by design (lines are bucketed by width, so the
  flattened chunk count and encoder length differ batch to batch). XLA compiles one
  graph per distinct shape, so expect a few hundred recompiles early in the run before
  the compilation cache covers the common shapes and step time settles.

**Platform settings to check first:**
- **Kaggle**: internet access is off by default -- Settings (right sidebar) > Internet >
  On. Without it, both `pip install` and the data-pull cell (which streams from the
  Hugging Face Hub) will fail. Also set Settings > Accelerator to GPU/TPU if you want
  one. Kaggle sessions have a disk budget too (check Settings) -- this notebook's
  ~9GB default pull should fit comfortably, but reduce `SAMPLES_PER_SOURCE` (section 2)
  if you're tight on space.
- **Colab**: Runtime > Change runtime type > pick GPU or TPU if you want one (default
  is CPU-only).


## This notebook: from-scratch comparison run

Companion to `train_recognizer_v2.ipynb`, which continues the existing `"v1"`
run from its latest checkpoint with three fixes applied partway through
training (from ~step 44,000 onward):

1. `ctc_weight`: 0.3 -> 0.5
2. `sequential_ar_steps`: 20,000 -> 60,000
3. Data-quality filter dropping transcripts with embedded newlines
   (image/text line-count mismatches -- see section 2c)

This notebook instead trains a **brand-new model from step 0** with all
three fixes active for the *entire* run, so you can compare two honest
endpoints against each other once both finish (or at matched step counts
along the way):

- `"v1"` (via `train_recognizer_v2.ipynb`): ~44k steps under the *old*
  config, fixes applied only for the remainder.
- `"v2_scratch"` (this notebook): fixes active from step 0 through however
  far it gets.

That tells you something the resumed run alone can't: whether the plateau
you saw was specifically caused by 44k steps of encoder weights already
shaped by the old 0.3/20,000/unfiltered-data config (in which case scratch
should clearly pull ahead), or whether the fixes work about as well applied
mid-run (in which case the two should converge to similar final numbers, and
the resumed run was the cheaper way to get there).

**Important -- this uses a separate run name AND a separate Hugging Face
repo** (`RUN_NAME = "v2_scratch"`, a new `CHECKPOINT_REPO_ID`), not just a
different local folder. `hf_push.push_checkpoint` uploads checkpoints to its
target repo by bare filename (`step_0044000.pt`, no run-name prefix) -- if
this notebook pushed to the same `Panhapich/tuna-ocr` repo `"v1"` uses, its
early checkpoints (step_0002000.pt, step_0044000.pt, ...) would silently
overwrite `"v1"`'s checkpoints at those exact step numbers, and once this
run's step count passes `"v1"`'s, `pull_latest_checkpoint` would start
resuming `"v1"` from the WRONG model without any error. Keeping them in
separate repos avoids all of that.

**Also important -- this is a second full training run, not a quick check.**
It runs up to the same `max_steps=200_000` budget as `"v1"`, on its own GPU
time, in parallel with (or instead of) the resumed run. Budget for that before
starting it.

In [1]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
def pull_latest():
    """Fast-forward the clone we're standing in, LOUDLY. A stale clone is the
    single most confusing failure mode of this notebook: the library code is
    older than the notebook cell driving it, so you get a TypeError about an
    unexpected keyword argument for a config field that plainly exists on
    GitHub. Failing to pull is survivable (offline runtime, dirty tree), so
    this doesn't raise -- but it must never be a quiet one-line note."""
    try:
        run_git(["pull", "--ff-only"])
        print("pulled latest changes")
    except RuntimeError as e:
        print("!" * 78)
        print("WARNING: could not update the clone -- running POSSIBLY STALE code.")
        print(f"  {e}")
        print("  If a later cell fails with 'unexpected keyword argument', this is why.")
        print("  Fix: !git -C . fetch origin && git -C . reset --hard origin/main")
        print("       then restart the runtime (stale modules stay imported).")
        print("!" * 78)

if os.path.isdir("recognizer"):
    # Already inside the repo -- which is what re-running this cell in the same
    # session always looks like, since the first run chdir'd here. This branch
    # used to just print and return, so a second run silently kept whatever code
    # the session started with and never saw upstream commits again.
    print(f"already inside the repo working dir: {os.getcwd()}")
    pull_latest()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    pull_latest()
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
# Print the resolved commit: the one unambiguous answer to "is my library code
# actually the version I think it is?", checkable against the GitHub history.
print("repo commit:  " + run_git(["log", "-1", "--pretty=%h %s"]).stdout.strip())


environment: colab
found an existing clone, reusing it: /content/tuna-ocr
pulled latest changes
working dir: /content/tuna-ocr
repo commit:  2218265 Truncate AR decodes at their real end, and stop scoring padding


In [2]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


using preinstalled torch 2.11.0+cu128 (cuda available: True) -- not reinstalled


In [3]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


platform secret store unavailable (RuntimeError), trying fallbacks...
environment:       colab
checkpoint root:   /content/drive/My Drive/tuna-ocr/checkpoints
HF token loaded:   True
accelerator:       cuda: Tesla T4 (15.6 GB)
torch device:      cuda


In [4]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetched Panhapich/khmer-sp-8k -> /content/tuna-ocr/recognizer/tokenizer/assets: ['gazetteer.json', 'khmer_segmentation.py', 'khmer_sp.model', 'latin_exceptions.json', 'tokenizer_info.json']


PosixPath('/content/tuna-ocr/recognizer/tokenizer/assets')

## 1b. Low-memory dataset loading patch

`recognizer/data/manifest.py`'s `load_dedup_arrow` (unmodified library code)
materializes the ENTIRE image-bytes column into a Python list
(`table.column("image").to_pylist()`), then builds a second full list of
`Sample` objects from it -- both lists stay alive simultaneously until the
function returns, so peak memory during dataset loading is roughly **2x**
the dataset's actual image-bytes size. On a Colab session this is enough to
get the kernel OOM-killed mid-load, which surfaces as no Python traceback at
all -- just `"Canceled future for execute_request message before replies
were done"` -- because the process itself dies, not one call inside it.

This cell monkeypatches `load_dedup_arrow` to iterate the Arrow columns
directly instead of pre-snapshotting them, so no intermediate full-column
Python list is ever alive alongside the final result -- same output, roughly
half the peak memory. This is a real fix to shared library code, not a
notebook-only workaround -- worth upstreaming into
`recognizer/data/manifest.py` directly once confirmed, so every consumer of
`run_training` benefits, not just this notebook.

In [5]:
# Monkeypatches recognizer.data.manifest.load_dedup_arrow: safe because
# load_dedup_manifest (which run_training actually calls) looks up
# load_dedup_arrow by name in the module's own namespace at CALL time, not at
# import time -- so reassigning the module attribute here takes effect for
# every call made after this cell runs, without needing to touch train.py or
# re-import anything downstream.
import recognizer.data.manifest as _manifest

def _load_dedup_arrow_low_memory(path):
    import pyarrow as pa

    with pa.memory_map(str(path), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    text_col = table.column("text")
    source_col = table.column("source")
    image_col = table.column("image")
    # zip() over ChunkedArrays iterates chunk-by-chunk, yielding pa.Scalar
    # objects one at a time -- .as_py() converts just that one value, so at
    # most one row's worth of extra Python objects exists beyond the `samples`
    # list actually being built, vs. the original's three full-column lists
    # PLUS the final list all alive at once.
    samples = []
    for t, s, img in zip(text_col, source_col, image_col):
        samples.append(_manifest.Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))
    return samples

_manifest.load_dedup_arrow = _load_dedup_arrow_low_memory
print("patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading")

patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading


## 2. Data

The full pull -> pack -> dedup pipeline below is expensive (real network transfer +
CPU-bound hashing, potentially a long time at this scale) -- it only needs to run
**once**. The first successful run pushes its result to a private Hugging Face
dataset repo (`real_data.config.HF_DATA_REPO_ID`); every later run (new session, new
notebook, different machine) checks that repo first and just downloads the prebuilt
`dedup.arrow` instead of repeating the pull/pack/dedup work from scratch.

`SAMPLES_PER_SOURCE` only matters the first time (before anything's been pushed to the
Hub). The defaults pull everything available from the three smaller sources, but cap
`chanrith_ocr_image_line` (12M+ rows, ~40GB in full) at 100k rows. Each source is
pulled, packed into a single Arrow file (`<source>.arrow`, image bytes stored exactly
as pulled -- no re-encoding/resizing), and its raw per-image files are deleted before
the next source starts, bounding peak disk usage to "one source's raw files + all
Arrow files packed so far."


In [6]:
import os, shutil, subprocess, sys
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, HF_DATA_REPO_ID, REAL_DATA_ROOT
from real_data import hf_push

# Per-source sample counts for the (one-time) pull from source. Pulls everything
# available from the smaller sources, but caps chanrith_ocr_image_line (12M+ rows) at
# 100k -- pulling it in full would be ~40GB, far more than a Kaggle/Colab session's
# disk budget can hold.
SAMPLES_PER_SOURCE = {
    "deepcopy_khmer_text_recognition": 136_117,
    "chanrith_ocr_image_line": 100_000,
    "darayut_scene_text": 102_500,
    "sokheng_synthetic_v1": 100_000,
}

# KMP_DUPLICATE_LIB_OK/OMP_NUM_THREADS: Colab/Kaggle commonly have more than one
# OpenMP runtime on the import path (numpy, PIL/imagehash, datasets' native deps each
# bundle their own libomp/libiomp5) -- loading two in one process is a well-known cause
# of an immediate SIGABRT with zero output, right at import time, before any of this
# script's own code runs. Setting these before spawning avoids that class of crash;
# harmless if it wasn't actually the cause.
SUBPROCESS_ENV = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1"}

def run_checked(cmd):
    """Runs `cmd`, always printing its output, and raises with the actual captured
    stderr on failure -- a bare `subprocess.CalledProcessError` (or, worse, a `!shell`
    cell whose exit code isn't checked at all) hides exactly the text that explains
    *why* it died, which is the difference between a one-line fix and a guessing game."""
    result = subprocess.run(cmd, env=SUBPROCESS_ENV, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        sys.stderr.write(result.stderr)
        raise RuntimeError(
            f"command failed (exit code {result.returncode}"
            f"{', likely killed by a signal -- see stderr above for the real cause' if result.returncode < 0 else ''}"
            f"): {' '.join(cmd)}"
        )
    return result

dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup.arrow"
built_locally = False  # tracks whether THIS run built dedup_manifest from source
                        # (vs. it already being local, or downloaded from the Hub) --
                        # only push to the Hub in the first case (section 2b below).

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping pull/download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"found a prebuilt dataset on the Hub ({HF_DATA_REPO_ID}) -- downloading "
          f"instead of re-pulling/re-deduplicating from source")
    hf_push.pull_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID)
else:
    print(f"no prebuilt dataset found on {HF_DATA_REPO_ID} -- pulling + packing from "
          f"source (one-time cost; result gets pushed to the Hub in the next cell)")
    built_locally = True

    # Pull -> pack to Arrow -> delete raw, one source at a time (not all sources
    # pulled first, then packed): this bounds peak disk usage to "current source's
    # raw files + every Arrow file packed so far", instead of needing all 4 sources'
    # raw files on disk simultaneously.
    arrow_files = []
    for source in EXTERNAL_DATASETS:
        arrow_path = REAL_DATA_ROOT / "samples" / f"{source}.arrow"
        arrow_files.append(arrow_path)
        if arrow_path.exists():
            print(f"{source}: already packed, skipping")
            continue

        source_dir = REAL_DATA_ROOT / "samples" / source
        num_samples = SAMPLES_PER_SOURCE[source]
        if not (source_dir / "manifest.tsv").exists():
            print(f"{source}: pulling {num_samples} samples...")
            run_checked([sys.executable, "-m", "real_data.generate_external_chunks",
                         "--source", source, "--num-samples", str(num_samples)])

        print(f"{source}: packing to {arrow_path}...")
        run_checked([sys.executable, "-m", "real_data.pack_arrow",
                     "--source", source, "--delete-raw"])

    print(arrow_files)


/content/tuna-ocr/real_data/samples/dedup.arrow already present locally, skipping pull/download


In [7]:
# 2b. Deduplicate + push to the Hub -- only runs if this session actually built the
# dataset from source above (built_locally == True); a no-op if dedup_manifest was
# already local or was just downloaded from the Hub.
if built_locally:
    # --near-dup-threshold 0 disables the O(n^2) near-dup pass -- REQUIRED at this
    # scale (hundreds of thousands of rows): the default pairwise comparison is
    # O(n^2) and would take an impractically long time (the nonzero default is only
    # tuned/safe for the notebook-scale hundreds-to-thousands range, e.g.
    # notebooks/train_diagnose_eval.ipynb).
    missing = [str(p) for p in arrow_files if not p.exists()]
    if missing:
        raise RuntimeError(
            "The following sources are missing their packed .arrow file -- re-run "
            "the pull cell above (in full, for all 4 sources) before deduplicating. "
            "This usually means the runtime restarted/reset between the pull and "
            "dedup cells (e.g. after a crash) and the previously-pulled data under "
            f"{REAL_DATA_ROOT} was lost:\n  " + "\n  ".join(missing)
        )

    run_checked([sys.executable, "-m", "real_data.deduplicate",
                 "--arrow-files", *[str(p) for p in arrow_files],
                 "--out", str(dedup_manifest),
                 "--near-dup-threshold", "0"])
    assert dedup_manifest.exists(), (
        f"{dedup_manifest} was not created -- check the pull cell above actually "
        f"populated {[str(p) for p in arrow_files]} before dedup ran."
    )
    print("dedup arrow file ready:", dedup_manifest)

    print(f"pushing prebuilt dataset to the Hub ({HF_DATA_REPO_ID}) so future runs "
          f"can skip straight to downloading it...")
    url = hf_push.push_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID, private=True)
    print("pushed:", url)
else:
    print(f"{dedup_manifest} already ready (local or from the Hub) -- nothing to dedup/push")


/content/tuna-ocr/real_data/samples/dedup.arrow already ready (local or from the Hub) -- nothing to dedup/push


## 2c. Data quality filter

One of the three fixed diagnostic samples tracked during a prior training run
(ground truth `'វិរាគចិត្ត ២០០២៛\nជោះ 8x10,000'`, with a literal newline in the
transcript) produced garbage predictions across 8,000+ straight logged steps --
a strong sign the image is a single cropped line but the transcript spans two,
not something any amount of training fixes. `find_unlearnable` (run inside
`run_training`, next section) only drops samples whose CTC target is longer
than the encoder frames the image can produce -- it does not catch this
different failure mode, where the transcript simply does not correspond to
what's pictured.

This cell scans `dedup.arrow` for embedded newlines (the cheapest,
highest-confidence signal available without per-sample manual review) and
writes a filtered copy, `dedup_filtered.arrow`, that the training cell uses
instead. Runs once and is cached like every other data-prep step in this
notebook -- if you add other mismatch heuristics later, delete
`dedup_filtered.arrow` to force a rebuild.

In [8]:
import pyarrow as pa

dedup_filtered = dedup_manifest.parent / "dedup_filtered.arrow"

# Validity, not just existence: a prior interrupted write (Colab disconnect,
# kernel restart, out-of-memory mid-write) can leave a truncated/corrupt file
# behind, and pa.ipc.open_file on a corrupt file raises "ArrowInvalid: Not an
# Arrow file" much later, inside run_training -- confusing, since by then it
# looks like a training-cell bug rather than a leftover bad file from this
# cell. Checking exists() alone (as an earlier version of this cell did) trusts
# that leftover file forever, since it never gets rewritten once present.
def _is_valid_arrow_file(path):
    if not path.exists():
        return False
    try:
        with pa.memory_map(str(path), "rb") as f:
            pa.ipc.open_file(f).schema
        return True
    except pa.ArrowInvalid:
        return False

if _is_valid_arrow_file(dedup_filtered):
    print(f"{dedup_filtered} already present and valid, skipping filter pass")
else:
    if dedup_filtered.exists():
        print(f"{dedup_filtered} exists but is not a valid Arrow file "
              f"(likely an interrupted write from a previous session) -- rebuilding")
    print(f"scanning {dedup_manifest} for transcript/image line-count mismatches...")
    with pa.memory_map(str(dedup_manifest), "rb") as source:
        table = pa.ipc.open_file(source).read_all()

    texts = table.column("text").to_pylist()
    sources = table.column("source").to_pylist()

    # A literal "\n" in a transcript is the cheapest, highest-confidence signal
    # that the label spans more lines than the (single-line-cropped) image
    # actually shows -- no gradient update can fix a target that doesn't match
    # its image. Extend this predicate if other mismatch patterns turn up.
    keep_mask = [("\n" not in t) for t in texts]
    dropped_by_source = {}
    for t, s, keep in zip(texts, sources, keep_mask):
        if not keep:
            dropped_by_source[s] = dropped_by_source.get(s, 0) + 1

    n_total = len(texts)
    n_dropped = n_total - sum(keep_mask)
    print(f"dropping {n_dropped}/{n_total} samples with embedded newlines "
          f"(likely multi-line transcript vs single-line image): {dropped_by_source or 'none'}")

    filtered_table = table.filter(pa.array(keep_mask))
    # Write to a .tmp path and atomically rename into place on success -- same
    # pattern dataset.py's compute_widths uses for its widths_cache.json, so an
    # interrupted write (Colab disconnect, OOM, kernel restart) never leaves a
    # half-written dedup_filtered.arrow sitting at the real filename for a later
    # run to mistake for a finished, valid file.
    tmp_path = dedup_filtered.with_name(dedup_filtered.name + ".tmp")
    with pa.OSFile(str(tmp_path), "wb") as sink:
        with pa.ipc.new_file(sink, filtered_table.schema) as writer:
            writer.write_table(filtered_table)
    tmp_path.replace(dedup_filtered)
    print(f"wrote filtered dataset -> {dedup_filtered}")

dedup_manifest = dedup_filtered
print("training will use:", dedup_manifest)

/content/tuna-ocr/real_data/samples/dedup_filtered.arrow already present and valid, skipping filter pass
training will use: /content/tuna-ocr/real_data/samples/dedup_filtered.arrow


## Rewind: extending the sequential stage to 60,000 steps

The original `v2_scratch` run switched from sequential to blockwise AR training at
step 40,000 (`sequential_ar_steps=40_000`), then hit a CUDA OOM at step 47,300 --
**unrelated to the switch itself** (that happened 7,300 steps earlier and had been
running stably in blockwise mode since): CUDA allocator fragmentation accumulating
over many steps of variable-shaped batches (width-bucketed, so `ar_logits`' size
differs batch to batch), evidenced by `1.21 GiB reserved by PyTorch but unallocated`
in the traceback. The startup batch-size probe already tests blockwise mode's worst
case at the widest images, so this wasn't a peak-memory miss -- it built up over the
run.

Rather than just resume the crashed curriculum forward from its last checkpoint, this
notebook rewinds to the step-40,000 checkpoint (the exact point of the original
switch -- confirmed still fully sequential: `ar_mode` is decided *before* the step
that produces the checkpoint, so `block_head` had received zero gradient updates at
that point) and extends the sequential stage another 20,000 steps --
`sequential_ar_steps=60_000` -- before switching to blockwise, in case the trunk's
cross-attention alignment hadn't plateaued by 40,000.

**`step_0042000.pt`/`0044000.pt`/`0046000.pt` -- trained under the original,
uncorrected `sequential_ar_steps=40_000` curriculum -- have been manually deleted
from `Panhapich/tuna-ocr-v2-scratch`.** `step_0040000.pt` is now the highest
checkpoint on that repo, so the training cell below uses plain "resume from latest"
with no special-cased rewind logic: there's nothing left on the repo that could be
ambiguously mistaken for the corrected curriculum's own progress.


In [ ]:
from pathlib import Path

import torch

from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training
from recognizer.hf_push import pull_latest_checkpoint

model_cfg = ModelConfig()
# Knobs worth touching from here rather than editing library code in Colab.
# LOG_EVERY: 100 gives NO output at all until step 100 completes, which is
#   indistinguishable from a hang on a cold start (XLA especially, where the first
#   steps pay graph compilation). Drop it to 1-10 for the first minutes of a new run
#   to confirm steps are actually advancing, then raise it back.
LOG_EVERY = 100
# NUM_WORKERS: 0 is the safe default *on CUDA* (see below). On a TPU or CPU runtime
#   there is no CUDA context to fork after, so 4 is safe there and takes the ~47ms/batch
#   of image-decode + tokenize off the critical path.
NUM_WORKERS = 0
# CKPT_EVERY: steps between checkpoint saves (and Hub pushes). The library
#   default of 10_000 is ~2h of training between saves at the measured
#   ~0.53s/step -- a lot to lose to a Colab disconnect. last.pt is written on
#   the same schedule, so a crash loses everything since the previous save.
CKPT_EVERY = 2_000

train_cfg = TrainConfig(
    log_every=LOG_EVERY,
    num_workers=NUM_WORKERS,
    ckpt_every=CKPT_EVERY,  # DataLoader workers fork *after* CUDA is initialized in a
                    # notebook kernel, which is unsafe and can deadlock silently (GPU
                    # pinned at 100% with zero real steps). 0 = safe synchronous loading.
    max_eval_samples=512,     # Val samples the periodic eval covers for CTC CER. Cheap:
                    # an argmax over an encoder pass, ~26ms/sample all-in.
    max_ar_eval_samples=64,   # ...but the AR greedy decode inside that eval emits ONE
                    # TOKEN PER FORWARD PASS in sequential mode. Measured on a real run:
                    # 512 AR decodes = 639s (10.7 min) every eval_every=500 steps, against
                    # 265s of actual training -- 71% of wall clock. 64 keeps the signal
                    # without owning the run; 0 = CTC CER only.
    ctc_weight=0.5,  # Same fix as train_recognizer_v2.ipynb, but active from step 0 here
                    # instead of partway through -- see this notebook's intro cell for why
                    # that distinction is the whole point of this run.
    sequential_ar_steps=60_000,  # Raised from 40_000 (see "Rewind" markdown cell just
                                 # above): the original v2_scratch run switched to
                                 # blockwise at 40,000 and then OOM'd at step 47,300
                                 # (unrelated to the switch -- CUDA allocator
                                 # fragmentation over many steps of variable-shaped
                                 # batches, see that cell). This restarts sequential
                                 # training from the step-40,000 checkpoint and extends
                                 # it another 20,000 steps before switching, in case the
                                 # trunk's alignment learning hadn't plateaued by 40,000
                                 # -- see recognizer/modules/decoder.py's "Two-stage
                                 # training" note for why this matters at all.
    max_steps=200_000,  # Explicit, matching the library default -- same budget as
                        # the original v2_scratch run and "v1", so all three stay
                        # comparable against the same LR schedule length.
)  # log_every=100, ckpt_every=10_000, ctc_weight=0.3, sequential_ar_steps=0 by default

# Separate identity from "v1" -- see this notebook's intro cell for why sharing the
# run name or Hub repo with "v1" would be actively harmful, not just confusing
# (checkpoints aren't namespaced by run_name on the Hub side).
RUN_NAME = "v2_scratch"
CHECKPOINT_REPO_ID = "Panhapich/tuna-ocr-v2-scratch"

# step_0042000.pt/0044000.pt/0046000.pt (trained under the original
# sequential_ar_steps=40_000 curriculum, before the step-47,300 OOM) have been deleted
# from this repo -- see the "Rewind" markdown cell above. step_0040000.pt is now the
# highest checkpoint on the HUB, so pulling from the Hub resolves correctly with no
# ambiguity. But a LOCAL last.pt is a separate risk the Hub deletion doesn't cover:
# the original crashed run wrote its own local last.pt on the same ckpt_every cadence,
# and that file was never touched by the manual Hub cleanup above. If this cell runs
# again on the same machine/Drive that ran the original attempt, resuming from a local
# last.pt trained past step 40,000 would silently pick up the uncorrected curriculum's
# weights -- exactly what deleting the Hub checkpoints was meant to prevent, just via a
# path that deletion doesn't reach. So: peek the local checkpoint's own step before
# trusting it, and refuse anything past the rewind point.
REWIND_STEP = 40_000  # step_0040000.pt is the last checkpoint valid under BOTH the
                       # original sequential_ar_steps=40_000 curriculum and this run's
                       # corrected sequential_ar_steps=60_000 one.

def _peek_checkpoint_step(path):
    return torch.load(path, map_location="cpu", weights_only=False)["step"]

local_last = Path(checkpoint_root) / RUN_NAME / "last.pt"
if local_last.exists():
    local_step = _peek_checkpoint_step(local_last)
    if local_step > REWIND_STEP:
        print(f"local checkpoint {local_last} is at step {local_step}, past the rewind "
              f"point of {REWIND_STEP} -- it was trained under the OLD, uncorrected "
              f"sequential_ar_steps=40_000 curriculum (before this cell's rewind to "
              f"60_000). Ignoring it and falling back to the Hub instead of silently "
              f"resuming from stale/wrong weights.")
        local_last = None
if local_last is not None and local_last.exists():
    resume_path = local_last
    print(f"resuming from local checkpoint: {resume_path}")
else:
    resume_path = pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                         repo_id=CHECKPOINT_REPO_ID)
    if resume_path:
        print(f"no usable local checkpoint -- resuming from the latest one on the Hub: {resume_path}")
    else:
        print(f"no local or Hub checkpoint found for {CHECKPOINT_REPO_ID} -- starting a fresh run")

# CHECKPOINT_REPO_ID above is a NEW repo (not Panhapich/tuna-ocr) -- created private by
# default the first time a checkpoint is pushed (hub_private=True) -- set to False only
# if you've deliberately decided this comparison repo should be public.
model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,  # points at dedup_filtered.arrow -- see section 2c;
                                          # same filtered dataset "v1" now trains on, so the
                                          # comparison isn't confounded by different data.
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune. CUDA only -- on TPU/CPU the
                       # configured TrainConfig.batch_size is used as-is, since XLA
                       # compiles lazily and never raises a catchable Python OOM.
    resume_path=resume_path,
)

## 4. Compare against the resumed run

Both runs write `eval_log.csv` (columns: `step,epoch,val_ar_cer,val_ctc_cer`)
to `checkpoint_root/<run_name>/`, and both share the same `checkpoint_root`
in this environment (only `run_name` differs), so both logs are already
sitting next to each other once you've run some steps of each. This cell
loads both and prints the best CER each has reached so far, plus a
step-matched comparison -- the fairest read, since both runs use the same
`max_steps=200_000` LR schedule, so the same step number means the same
position in that schedule for either run.

Run this anytime -- it only reads log files, so it works mid-run for a
progress check, not just at the end.

In [ ]:
import pandas as pd

RUNS_TO_COMPARE = {
    "v1 (resumed, fixes from ~44k)": "v1",
    "v2_scratch (fixes from step 0)": "v2_scratch",
}

logs = {}
for label, run_name in RUNS_TO_COMPARE.items():
    path = Path(checkpoint_root) / run_name / "eval_log.csv"
    if not path.exists():
        print(label + ": no eval_log.csv yet at " + str(path) + " -- has this run started?")
        continue
    df = pd.read_csv(path)
    logs[label] = df
    best_ctc_row = df.loc[df["val_ctc_cer"].idxmin()]
    best_ar_row = df.loc[df["val_ar_cer"].idxmin()]
    latest_step = int(df["step"].iloc[-1])
    print(f"{label}: {len(df)} eval points, latest step {latest_step}")
    print(f"  best val_ctc_cer: {best_ctc_row.val_ctc_cer:.4f} at step {int(best_ctc_row.step)}")
    print(f"  best val_ar_cer:  {best_ar_row.val_ar_cer:.4f} at step {int(best_ar_row.step)}")
    print()

if len(logs) == 2:
    (label_a, df_a), (label_b, df_b) = logs.items()
    max_common_step = min(df_a["step"].max(), df_b["step"].max())
    print(f"step-matched comparison up to step {int(max_common_step)} "
          "(the furthest both runs have reached):")
    for label, df in ((label_a, df_a), (label_b, df_b)):
        window = df[df["step"] <= max_common_step]
        last_row = window.iloc[-1]
        print(f"  {label}: val_ctc_cer {last_row.val_ctc_cer:.4f}, "
              f"val_ar_cer {last_row.val_ar_cer:.4f} (at step {int(last_row.step)})")